# Notebook 05: City-Wide Pipeline Evaluation
**Multi-Camera Tracking System — CityFlowV2 (Vehicles) + WILDTRACK (People)**

This notebook downloads two real-world multi-camera datasets and runs the full
7-stage MTMC pipeline on each, validating both the **vehicle** and **person**
tracking paths at proper surveillance resolution.

| Dataset | Objects | Cameras | Resolution | Size |
|---------|---------|---------|------------|------|
| CityFlowV2 (AIC 2022) | Vehicles (880 IDs) | 46 across 16 intersections | 960p+ | ~15–20 GB |
| WILDTRACK (EPFL) | People (~20/frame) | 7 overlapping | 1920×1080 | ~6.3 GB |

**Runtime**: GPU (P100/T4) required | **Duration**: 2–4 hours total | **Disk**: ~40 GB

---
## 0. Environment Setup

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

# --- CONFIG: Set your repo source ---
# Option A: Clone from GitHub (set your repo URL)
REPO_URL = ""  # e.g. "https://github.com/youruser/mtmc-tracker.git"

# Option B: Upload repo as Kaggle dataset and set path here
REPO_DATASET = ""  # e.g. "mrkdagods/mtmc-tracker-source"

# Paths
WORK_DIR = Path("/kaggle/working")
PROJECT_DIR = WORK_DIR / "mtmc-tracker"
DATA_DIR = PROJECT_DIR / "data" / "raw"
MODELS_DIR = PROJECT_DIR / "models"

print(f"Work dir: {WORK_DIR}")
print(f"GPU available: {subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()}")

# Check disk space
import shutil as _sh
total, used, free = _sh.disk_usage("/kaggle/working")
print(f"Disk: {free / (1024**3):.1f} GB free / {total / (1024**3):.1f} GB total")

In [ ]:
# Clone the project
if PROJECT_DIR.exists():
    print(f"Project already exists at {PROJECT_DIR}, pulling latest...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)
else:
    print(f"Cloning from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print(f"Working directory: {os.getcwd()}")
print(f"Contents: {[p.name for p in PROJECT_DIR.iterdir()]}")

In [ ]:
# Install dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q ultralytics boxmot faiss-gpu opencv-python-headless \
    omegaconf loguru networkx scikit-learn plotly pandas click rich gdown tqdm

In [ ]:
# Verify GPU and key imports
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / (1024**3):.1f} GB")

import cv2
import numpy as np
import faiss
print(f"OpenCV: {cv2.__version__}")
print(f"FAISS GPU support: {faiss.get_num_gpus() > 0}")
print("All imports OK.")

---
## 1. Download Datasets

Downloads both datasets to `data/raw/`. CityFlowV2 comes from Google Drive (~15–20 GB),
WILDTRACK from EPFL's servers (~6.3 GB).

In [ ]:
# Download WILDTRACK first (smaller, direct HTTP, faster)
!python scripts/download_datasets.py --dataset wildtrack

In [ ]:
# Download CityFlowV2 (larger, Google Drive)
!python scripts/download_datasets.py --dataset cityflowv2

In [ ]:
# Verify datasets are in place
for ds_name, ds_path in [("cityflowv2", DATA_DIR / "cityflowv2"), ("wildtrack", DATA_DIR / "wildtrack")]:
    if ds_path.exists():
        contents = list(ds_path.iterdir())
        size_gb = sum(f.stat().st_size for f in ds_path.rglob("*") if f.is_file()) / (1024**3)
        print(f"{ds_name}: {len(contents)} items, {size_gb:.1f} GB")
    else:
        print(f"{ds_name}: NOT FOUND at {ds_path}")

# Disk check
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\nDisk remaining: {free / (1024**3):.1f} GB free")

---
## 2. Prepare Datasets

Verify directory structures and generate ground-truth files in MOT format.

In [ ]:
# Prepare CityFlowV2 (verify structure, list cameras)
!python scripts/prepare_dataset.py --dataset cityflowv2 --root data/raw/cityflowv2

In [ ]:
# Prepare WILDTRACK (convert JSON annotations to MOT format)
!python scripts/prepare_dataset.py --dataset wildtrack --root data/raw/wildtrack

---
## 3. Download Model Weights

If you trained ReID models in Notebooks 02–04, upload them as a Kaggle dataset
and copy them here. Otherwise the pipeline will use the weights bundled in the repo.

In [ ]:
# Check which model weights are available
model_files = {
    "YOLO detector": MODELS_DIR / "detection" / "yolo26m.pt",
    "Tracker ReID": MODELS_DIR / "tracker" / "osnet_x0_25_msmt17.pt",
    "Person ReID (ResNet50-IBN)": MODELS_DIR / "reid" / "person_resnet50ibn_market1501.pth",
    "Person ReID (OSNet)": MODELS_DIR / "reid" / "person_osnet_market1501.pth",
    "Vehicle ReID (ResNet50-IBN)": MODELS_DIR / "reid" / "vehicle_resnet50ibn_veri776.pth",
    "Vehicle ReID (OSNet)": MODELS_DIR / "reid" / "vehicle_osnet_veri776.pth",
}

for name, path in model_files.items():
    status = "OK" if path.exists() else "MISSING"
    size = f"({path.stat().st_size / (1024**2):.1f} MB)" if path.exists() else ""
    print(f"  [{status}] {name}: {path.name} {size}")

In [ ]:
# --- Optional: Copy trained ReID weights from a Kaggle dataset ---
# If you uploaded your trained weights as a Kaggle dataset, uncomment and set:
#
# WEIGHTS_DATASET = "/kaggle/input/your-reid-weights"
# for src_file in Path(WEIGHTS_DATASET).glob("*.pth"):
#     dest = MODELS_DIR / "reid" / src_file.name
#     if not dest.exists():
#         shutil.copy2(src_file, dest)
#         print(f"Copied {src_file.name} -> {dest}")

# Download YOLO weights if missing (auto-downloads via ultralytics)
!python scripts/download_models.py --models-dir models/

---
## 4. Run Pipeline: CityFlowV2 (Vehicles, City-Scale)

46 cameras across 16 city intersections. This is the primary benchmark for
vehicle tracking at real surveillance resolution.

**Expected**: Detection of cars/buses/trucks, per-camera tracking, cross-camera
re-identification using vehicle ReID embeddings, global ID assignment.

In [ ]:
%%time
# Run full pipeline on CityFlowV2
# Using the cityflowv2 dataset config to override defaults
!python scripts/run_pipeline.py \
    --config configs/default.yaml \
    --dataset-config configs/datasets/cityflowv2.yaml \
    -o project.run_name=cityflowv2_eval \
    -o stage0.input_dir=data/raw/cityflowv2

In [ ]:
# Inspect CityFlowV2 results
import json

cfv2_output = PROJECT_DIR / "data" / "outputs" / "cityflowv2_eval"

# Stage 4 results: global trajectories
traj_path = cfv2_output / "stage4" / "global_trajectories.json"
if traj_path.exists():
    with open(traj_path) as f:
        trajectories = json.load(f)
    print(f"CityFlowV2 Results:")
    print(f"  Global identities: {len(trajectories)}")
    # Show top 10 largest identities
    items = list(trajectories.items()) if isinstance(trajectories, dict) else [(str(i), t) for i, t in enumerate(trajectories)]
    items.sort(key=lambda x: len(x[1].get('tracklets', x[1].get('tracklet_ids', [])) if isinstance(x[1], dict) else []), reverse=True)
    for gid, traj in items[:10]:
        if isinstance(traj, dict):
            tracklets = traj.get('tracklets', traj.get('tracklet_ids', []))
            n_tracklets = len(tracklets)
            cameras = set()
            for t in tracklets:
                if isinstance(t, dict):
                    cameras.add(t.get('camera_id', '?'))
            cam_str = f" across {len(cameras)} cameras ({', '.join(sorted(cameras))})" if cameras else ""
            print(f"  GID {gid}: {n_tracklets} tracklets{cam_str}")
else:
    print(f"No trajectories found at {traj_path}")
    print(f"Contents of output dir:")
    if cfv2_output.exists():
        for p in sorted(cfv2_output.rglob("*")):
            if p.is_file():
                print(f"  {p.relative_to(cfv2_output)}")

In [ ]:
# Stage 5: Evaluation metrics (if ground truth was available)
eval_path = cfv2_output / "stage5" / "evaluation_results.json"
if eval_path.exists():
    with open(eval_path) as f:
        metrics = json.load(f)
    print("CityFlowV2 Evaluation Metrics:")
    for metric, value in metrics.items():
        if isinstance(value, float):
            print(f"  {metric}: {value:.4f}")
        else:
            print(f"  {metric}: {value}")
else:
    print("No evaluation results (ground truth may not have been loaded).")
    print("Check stage5/ for details.")

---
## 5. Run Pipeline: WILDTRACK (People, HD)

7 overlapping 1080p cameras in an outdoor area. This validates the person
tracking path with much higher resolution than EPFL Lab (1920x1080 vs 360x288).

In [ ]:
%%time
# Run full pipeline on WILDTRACK
!python scripts/run_pipeline.py \
    --config configs/default.yaml \
    --dataset-config configs/datasets/wildtrack.yaml \
    -o project.run_name=wildtrack_eval \
    -o stage0.input_dir=data/raw/wildtrack/videos

In [ ]:
# Inspect WILDTRACK results
wt_output = PROJECT_DIR / "data" / "outputs" / "wildtrack_eval"

traj_path = wt_output / "stage4" / "global_trajectories.json"
if traj_path.exists():
    with open(traj_path) as f:
        trajectories = json.load(f)
    print(f"WILDTRACK Results:")
    print(f"  Global identities: {len(trajectories)}")
    for gid, traj in sorted(trajectories.items(), key=lambda x: len(x[1].get('tracklets', [])), reverse=True)[:10]:
        n_tracklets = len(traj.get('tracklets', []))
        cameras = set(t.get('camera_id', '?') for t in traj.get('tracklets', []))
        print(f"  GID {gid}: {n_tracklets} tracklets across {len(cameras)} cameras ({', '.join(sorted(cameras))})")
else:
    print(f"No trajectories found at {traj_path}")
    print(f"Contents of {wt_output}:")
    if wt_output.exists():
        for p in sorted(wt_output.rglob("*")):
            if p.is_file():
                print(f"  {p.relative_to(wt_output)}")

In [ ]:
# Stage 5: WILDTRACK evaluation metrics
eval_path = wt_output / "stage5" / "evaluation_results.json"
if eval_path.exists():
    with open(eval_path) as f:
        metrics = json.load(f)
    print("WILDTRACK Evaluation Metrics:")
    for metric, value in metrics.items():
        if isinstance(value, float):
            print(f"  {metric}: {value:.4f}")
        else:
            print(f"  {metric}: {value}")
else:
    print("No evaluation results found.")

---
## 6. Side-by-Side Comparison

In [ ]:
import pandas as pd

def load_run_summary(run_dir: Path, dataset_name: str) -> dict:
    """Extract key stats from a pipeline run."""
    summary = {"dataset": dataset_name}

    # Count tracklets from stage1
    stage1 = run_dir / "stage1"
    if stage1.exists():
        tracklet_files = list(stage1.glob("*_tracklets.json")) + list(stage1.glob("**/tracklets*.json"))
        summary["tracklet_files"] = len(tracklet_files)

    # Global trajectories from stage4
    traj_path = run_dir / "stage4" / "global_trajectories.json"
    if traj_path.exists():
        with open(traj_path) as f:
            trajs = json.load(f)
        summary["global_ids"] = len(trajs)
        all_tracklets = []
        all_cameras = set()
        for t in trajs.values():
            tl = t.get("tracklets", [])
            all_tracklets.extend(tl)
            for tr in tl:
                all_cameras.add(tr.get("camera_id", "?"))
        summary["total_tracklets"] = len(all_tracklets)
        summary["cameras_used"] = len(all_cameras)

    # Eval metrics from stage5
    eval_path = run_dir / "stage5" / "evaluation_results.json"
    if eval_path.exists():
        with open(eval_path) as f:
            metrics = json.load(f)
        for k in ["HOTA", "MOTA", "IDF1"]:
            if k in metrics:
                summary[k] = metrics[k]

    # Videos from stage6
    stage6 = run_dir / "stage6"
    if stage6.exists():
        videos = list(stage6.rglob("*.mp4"))
        summary["output_videos"] = len(videos)

    return summary

rows = []
for name, run_dir in [("CityFlowV2", cfv2_output), ("WILDTRACK", wt_output)]:
    if run_dir.exists():
        rows.append(load_run_summary(run_dir, name))

if rows:
    df = pd.DataFrame(rows).set_index("dataset")
    print(df.to_string())
else:
    print("No results to compare yet.")

---
## 7. Visualize Sample Results

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Video, display, HTML

def show_sample_frames(run_dir: Path, dataset_name: str, n_frames: int = 3):
    """Display sample annotated frames from stage6 videos."""
    vid_dir = run_dir / "stage6" / "videos"
    if not vid_dir.exists():
        print(f"No videos found for {dataset_name}")
        return

    videos = sorted(vid_dir.glob("*.mp4"))[:4]  # First 4 cameras
    if not videos:
        print(f"No MP4 files in {vid_dir}")
        return

    fig, axes = plt.subplots(len(videos), n_frames, figsize=(5*n_frames, 4*len(videos)))
    if len(videos) == 1:
        axes = [axes]

    for row, vpath in enumerate(videos):
        cap = cv2.VideoCapture(str(vpath))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        sample_indices = [int(total * i / (n_frames + 1)) for i in range(1, n_frames + 1)]

        for col, fidx in enumerate(sample_indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
            ret, frame = cap.read()
            ax = axes[row][col] if n_frames > 1 else axes[row]
            if ret:
                ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{vpath.stem} | frame {fidx}", fontsize=9)
            ax.axis("off")
        cap.release()

    plt.suptitle(f"{dataset_name} — Annotated Sample Frames", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(run_dir / f"{dataset_name.lower()}_samples.png", dpi=150, bbox_inches="tight")
    plt.show()

show_sample_frames(cfv2_output, "CityFlowV2")
show_sample_frames(wt_output, "WILDTRACK")

---
## 8. Package Results for Download

Zip up the key outputs (trajectories, metrics, annotated videos, timeline charts)
so you can download them from Kaggle and inspect locally.

In [ ]:
import zipfile

def package_results(run_dir: Path, dataset_name: str) -> Path:
    """Create a downloadable zip of pipeline outputs."""
    zip_path = WORK_DIR / f"{dataset_name}_results.zip"

    # Collect files to include (skip raw extracted frames to save space)
    include_patterns = [
        "stage4/*.json",       # global trajectories
        "stage5/*.json",       # evaluation metrics
        "stage5/*.html",       # evaluation report
        "stage6/videos/*.mp4", # annotated videos
        "stage6/*.html",       # timeline charts
        "stage6/exports/*",    # JSON/CSV exports
        "config.yaml",         # run config
        "pipeline.log",        # execution log
    ]

    if not run_dir.exists():
        print(f"Run directory not found: {run_dir}")
        return None

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for pattern in include_patterns:
            for f in run_dir.glob(pattern):
                if f.is_file():
                    arcname = f"{dataset_name}/{f.relative_to(run_dir)}"
                    zf.write(f, arcname)
                    print(f"  + {arcname} ({f.stat().st_size / (1024**2):.1f} MB)")

    if zip_path.exists():
        print(f"\nPackaged: {zip_path} ({zip_path.stat().st_size / (1024**2):.1f} MB)")
    return zip_path

print("=" * 60)
print("Packaging CityFlowV2 results...")
package_results(cfv2_output, "cityflowv2")

print("\n" + "=" * 60)
print("Packaging WILDTRACK results...")
package_results(wt_output, "wildtrack")

print("\n" + "=" * 60)
print("Download the zip files from the Output tab on the right.")

---
## 9. Analysis Notes

### What to look for in the results

**CityFlowV2 (Vehicles)**:
- Each global ID should represent one vehicle across multiple intersection cameras
- Vehicles that pass through the same intersection at different times should get *different* IDs
- Check annotated videos: do cars keep their ID as they move through an intersection?
- The vehicle ReID model (ResNet50-IBN-a trained on VeRi-776) should perform well at 960p+

**WILDTRACK (People)**:
- With 7 overlapping 1080p cameras and ~20 people, this is a much better test than EPFL Lab
- Person crops will be 100-300px tall (vs 30-80px in EPFL Lab) — ReID should work properly
- Cross-camera matching should be strong since all cameras share overlapping field-of-view
- The 7-camera overlap means the same person is visible from multiple angles simultaneously

### Key metrics to compare
| Metric | What it measures | Target |
|--------|-----------------|--------|
| HOTA | Balanced detection + association quality | > 40% |
| MOTA | Detection accuracy (FP, FN, ID switches) | > 50% |
| IDF1 | Identity preservation across cameras | > 50% |
| ID switches | How often an identity jumps to wrong person | Lower is better |